# 🦎 Iguana Species Classification — Transfer Learning Assignment

In this notebook we will classify **7 species of iguana** using Transfer Learning.

The 7 classes are:
- Black_spiny_tailed_iguana
- Brown_anole
- Cuban_knight_anole
- Desert_iguana
- Green_anole
- Green_iguana
- Lesser_Antillean_iguana

**Strategy:** We use a pretrained EfficientNetV2S model (trained on ImageNet) as a **feature extractor** with frozen weights, and add our own classification head on top.

---
### 📁 Expected folder structure
```
data/
  train/
    Black_spiny_tailed_iguana/  *.jpg
    Brown_anole/                *.jpg
    ...
  test/
    1.jpg, 2.jpg, ...
  train.csv               <- filename + label (for verification)
  test.csv                <- id column only (what we need to predict)
  Classes_nummering.xlsx  <- maps class names to numeric IDs
saved_models/             <- trained model will be saved here
```

## Step 0 — Install dependencies & Imports & GPU check

In [ ]:
# Install all required packages
# Run this cell once; you can skip it on subsequent runs if packages are already installed
!pip install tensorflow>=2.16 \
             numpy \
             pandas \
             matplotlib \
             scikit-learn \
             openpyxl \
             Pillow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {tf.config.list_physical_devices('GPU')}")

## Step 1 — Configuration

All the hyper-parameters you might want to tweak are collected here.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────────
DATA_DIR         = Path("data")
TRAIN_DIR        = DATA_DIR / "train"
TEST_DIR         = DATA_DIR / "test"
TRAIN_CSV        = DATA_DIR / "train.csv"
TEST_CSV         = DATA_DIR / "test.csv"
CLASSES_XLSX     = DATA_DIR / "Classes_nummering.xlsx"

SAVED_MODELS_DIR = Path("saved_models")
SAVED_MODELS_DIR.mkdir(exist_ok=True)   # create the folder if it doesn't exist yet
MODEL_PATH       = SAVED_MODELS_DIR / "iguana_model.keras"
SUBMISSION_PATH  = "submission.csv"

# ── image & training settings ──────────────────────────────────────────────────
IMG_SIZE         = 224       # EfficientNetV2S expects >= 224
BATCH_SIZE       = 32
EPOCHS           = 10
LEARNING_RATE    = 1e-3
VALIDATION_SPLIT = 0.2
SEED             = 42

NUM_CLASSES      = 7

## Step 2 — Explore the data

In [ ]:
# ── Read the class numbering ────────────────────────────────────────────────────
classes_df = pd.read_excel(CLASSES_XLSX)
print(classes_df)
print("\nColumns:", classes_df.columns.tolist())

In [ ]:
# ── Peek at the CSVs ────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train.csv shape:", train_df.shape)
print(train_df.head())
print("\ntest.csv shape:", test_df.shape)
print(test_df.head())

In [ ]:
# ── Count images per class ──────────────────────────────────────────────────────
class_counts = {}
for class_dir in sorted(TRAIN_DIR.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.png")))
        class_counts[class_dir.name] = count

print("Images per class:")
for cls, cnt in class_counts.items():
    print(f"  {cls:<40} {cnt}")
print(f"\nTotal training images: {sum(class_counts.values())}")

In [ ]:
# ── Visualize a few training images ────────────────────────────────────────────
fig, axes = plt.subplots(3, 7, figsize=(18, 8))
class_names = sorted(class_counts.keys())

for col, cls in enumerate(class_names):
    images = list((TRAIN_DIR / cls).glob("*.jpg"))[:3]
    for row, img_path in enumerate(images):
        img = plt.imread(img_path)
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(cls.replace("_", "\n"), fontsize=7)

plt.suptitle("Sample training images per species", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3 — Build the data pipeline

We use `image_dataset_from_directory` which automatically infers class labels from the subfolder names. **No manual one-hot encoding needed!**

We also apply **data augmentation** to reduce overfitting.

In [ ]:
# ── Training & validation datasets ─────────────────────────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",   # one-hot encoded -> use categorical_crossentropy
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

# The class names as discovered by Keras (alphabetical order)
CLASS_NAMES = train_ds.class_names
print("Class names (in model order):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")

In [ ]:
# ── Data augmentation (applied only during training) ───────────────────────────
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def prepare_train(images, labels):
    images = data_augmentation(images, training=True)
    return images, labels

def prepare_val(images, labels):
    return images, labels

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(prepare_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(prepare_val,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("Pipeline ready")

## Step 4 — Build the model

**Architecture:**
1. **Base model** — EfficientNetV2S pretrained on ImageNet, weights **frozen**
2. **Global Average Pooling** — collapses the feature maps into a single vector
3. **Our classification head** — Dense layers leading to a 7-class softmax output

> You can swap `EfficientNetV2S` for another backbone like `ResNet50V2` or `MobileNetV3Large` in one line.

In [ ]:
# ── Frozen base model (feature extractor) ──────────────────────────────────────
base_model = EfficientNetV2S(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,     # remove the original ImageNet classification head
    weights="imagenet",    # use pretrained ImageNet weights
)
base_model.trainable = False   # freeze all base layers

# ── Our classification head ────────────────────────────────────────────────────
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation="relu"),
    Dropout(0.4),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax"),   # softmax for multi-class
], name="iguana_classifier")

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Step 5 — Train the model

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
]

print(f"Training with frozen base for up to {EPOCHS} epochs...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# Save the model once after training is complete
model.save(MODEL_PATH)
print(f"\nModel saved to: {MODEL_PATH}")

## Step 6 — Visualize training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"],     label="train loss")
ax1.plot(history.history["val_loss"], label="val loss")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"],     label="train acc")
ax2.plot(history.history["val_accuracy"], label="val acc")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.suptitle("Training history — Feature Extraction (frozen body)", fontsize=14)
plt.tight_layout()
plt.show()

## Step 7 — Evaluate on validation set

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=1)
print(f"\nValidation accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")

In [ ]:
# ── Confusion matrix ────────────────────────────────────────────────────────────
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=[c.replace("_", "\n") for c in CLASS_NAMES])
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
plt.title("Confusion Matrix — Validation set")
plt.tight_layout()
plt.show()

## Step 8 — Generate submission file

Read the test images, predict, map class indices back to the numeric IDs from `Classes_nummering.xlsx`, and write `submission.csv`.

In [ ]:
# ── Load class numbering from xlsx ─────────────────────────────────────────────
# Inspect the columns printed in Step 2 and adjust the indices here if needed
classes_df = pd.read_excel(CLASSES_XLSX)

NAME_COL = classes_df.columns[0]   # column with the species name string
ID_COL   = classes_df.columns[1]   # column with the numeric id

name_to_id = dict(zip(classes_df[NAME_COL], classes_df[ID_COL]))
print("name -> id mapping:", name_to_id)

In [ ]:
# ── Read test ids ───────────────────────────────────────────────────────────────
test_df   = pd.read_csv(TEST_CSV)
ID_COLUMN = test_df.columns[0]   # usually 'id' or 'ID'
test_ids  = test_df[ID_COLUMN].tolist()
print(f"{len(test_ids)} test images to predict")

In [ ]:
# ── Predict on test images ──────────────────────────────────────────────────────
predictions = []

for img_id in test_ids:
    for ext in [".jpg", ".jpeg", ".png"]:
        img_path = TEST_DIR / f"{img_id}{ext}"
        if img_path.exists():
            break

    img       = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.expand_dims(tf.keras.utils.img_to_array(img), axis=0)

    pred       = model.predict(img_array, verbose=0)
    class_name = CLASS_NAMES[np.argmax(pred[0])]
    predictions.append(name_to_id[class_name])

print(f"Done! {len(predictions)} predictions made.")
print("Sample:", predictions[:10])

In [ ]:
# ── Write submission.csv ────────────────────────────────────────────────────────
submission_df = pd.DataFrame({ID_COLUMN: test_ids, "label": predictions})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to: {SUBMISSION_PATH}")
print(submission_df.head(10))

---
## Things to experiment with

Once the baseline is working, here are knobs you can turn to improve accuracy:

| What | Where | Try |
|------|-------|-----|
| **Base model** | Step 4 | Swap `EfficientNetV2S` -> `ResNet50V2`, `MobileNetV3Large`, `ConvNeXtBase` |
| **Image size** | Step 1 config | 224 -> 300 or 384 (bigger = more detail, more memory) |
| **Augmentation** | Step 3 | Add `RandomFlip("vertical")`, `RandomTranslation`, `RandomShear` |
| **Head architecture** | Step 4 | More/fewer Dense layers, different Dropout rates |
| **Epochs** | Step 1 config | Increase `EPOCHS` |
| **Fine-tuning** | After Step 5 | Set `base_model.trainable = True`, recompile with `lr=1e-5`, train again |
| **Class weights** | `model.fit()` | Add `class_weight=...` if classes are imbalanced |
| **Batch size** | Step 1 config | Smaller = noisier gradients (sometimes helps generalization) |